# Segmentation FP Fix — Threshold + Post-Processing
Use existing best model (97.62% val IoU), optimize for false positives

In [1]:
import torch
assert torch.cuda.is_available()
print(f'GPU: {torch.cuda.get_device_name(0)}')
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

GPU: Tesla T4
Mounted at /content/drive


In [2]:
!pip install -q segmentation-models-pytorch albumentations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 11.4 MB/s eta 0:00:00


In [4]:
from pathlib import Path

WORK_DIR = Path('/content')
DATA_DIR = WORK_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = WORK_DIR / 'checkpoints'
PLOTS_DIR = WORK_DIR / 'plots'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_DIR = Path('/content/drive/MyDrive/HeritagePreservation')

print(f"Paths setup OK")

Paths setup OK


In [5]:
import zipfile, shutil
from pathlib import Path

def extract_dataset(zip_name, data_dir, drive_dir):
    """Copy zip to local /content first, then extract — avoids Drive FUSE drops on large files."""
    zip_path = drive_dir / zip_name
    out_name = zip_name.replace('.zip', '')
    out_dir  = data_dir / out_name

    if not zip_path.exists():
        print(f'SKIP {zip_name} — not on Drive')
        return
    if out_dir.exists() and any(out_dir.rglob('*.*')):
        print(f'{zip_name}: already extracted, skipping.')
        return

    size_mb   = zip_path.stat().st_size / 1e6
    local_zip = Path(f'/content/_tmp_{out_name}.zip')

    print(f'Copying {zip_name} ({size_mb:.0f} MB) to local disk...')
    shutil.copy2(zip_path, local_zip)
    print(f'Extracting...')
    with zipfile.ZipFile(local_zip, 'r') as zf:
        zf.extractall(data_dir)
    local_zip.unlink()
    print(f'  Done.')


for zip_name in ['masonry.zip', 'crackforest.zip', 'historical_crack.zip', 'omnicrack30k.zip']:
    extract_dataset(zip_name, DATA_DIR, DRIVE_DIR)

print('All datasets extracted.')


Copying masonry.zip (21 MB) to local disk...
Extracting...
  Done.
Copying crackforest.zip (4 MB) to local disk...
Extracting...
  Done.
Copying historical_crack.zip (17 MB) to local disk...
Extracting...
  Done.
Copying omnicrack30k.zip (10604 MB) to local disk...
Extracting...
  Done.
All datasets extracted.


In [6]:
import cv2
import numpy as np
import random
from collections import Counter
from sklearn.model_selection import train_test_split

MAX_OMNI_SAMPLES = 3000

IMG_EXTS       = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
IMG_DIR_NAMES  = {'images', 'img', 'image', 'jpegimages', 'rgb', 'data'}
MASK_DIR_NAMES = {'masks', 'mask', 'labels', 'label', 'annotations', 'gts', 'gt'}


def find_image_mask_pairs(root_dir, max_samples=None, seed=42):
    root  = Path(root_dir)
    pairs = []
    seen  = set()
    for img_dir in root.rglob('*'):
        if not img_dir.is_dir() or img_dir.name.lower() not in IMG_DIR_NAMES:
            continue
        parent   = img_dir.parent
        mask_dir = None
        for mn in MASK_DIR_NAMES:
            c = parent / mn
            if c.is_dir():
                mask_dir = c
                break
        if mask_dir is None:
            continue
        for img_path in sorted(img_dir.glob('*.*')):
            if img_path.suffix.lower() not in IMG_EXTS:
                continue
            key = str(img_path)
            if key in seen:
                continue
            for ext in ['.png', '.jpg', '.bmp', img_path.suffix]:
                mask_path = mask_dir / f'{img_path.stem}{ext}'
                if mask_path.exists():
                    pairs.append((img_path, mask_path))
                    seen.add(key)
                    break
    if max_samples and len(pairs) > max_samples:
        rng = random.Random(seed)
        rng.shuffle(pairs)
        pairs = pairs[:max_samples]
    return pairs


masonry_pairs     = find_image_mask_pairs(DATA_DIR / 'masonry')
crackforest_pairs = find_image_mask_pairs(DATA_DIR / 'crackforest')
historical_pairs  = find_image_mask_pairs(DATA_DIR / 'historical_crack') \
                    if (DATA_DIR / 'historical_crack').exists() else []

# omnicrack30k: files in subdirs inside images/ and annotations/
omni_img_dir  = DATA_DIR / 'images'
omni_mask_dir = DATA_DIR / 'annotations'
omni_pairs    = []
if omni_img_dir.exists() and omni_mask_dir.exists():
    for img_path in sorted(omni_img_dir.rglob('*.*')):
        if img_path.suffix.lower() not in IMG_EXTS:
            continue
        rel = img_path.relative_to(omni_img_dir)
        for ext in ['.png', img_path.suffix, '.jpg']:
            mask_path = omni_mask_dir / rel.parent / f'{img_path.stem}{ext}'
            if mask_path.exists():
                omni_pairs.append((img_path, mask_path))
                break
    print(f'omnicrack30k found: {len(omni_pairs)} pairs')
    if len(omni_pairs) > MAX_OMNI_SAMPLES:
        rng = random.Random(42)
        rng.shuffle(omni_pairs)
        omni_pairs = omni_pairs[:MAX_OMNI_SAMPLES]
else:
    omni_pairs = find_image_mask_pairs(DATA_DIR / 'omnicrack30k', max_samples=MAX_OMNI_SAMPLES) \
                 if (DATA_DIR / 'omnicrack30k').exists() else []

all_pairs = masonry_pairs + crackforest_pairs + historical_pairs + omni_pairs
print(f'masonry       : {len(masonry_pairs)}')
print(f'crackforest   : {len(crackforest_pairs)}')
print(f'historical    : {len(historical_pairs)}')
print(f'omnicrack30k  : {len(omni_pairs)}')
print(f'Total         : {len(all_pairs)}')

random.seed(42)
train_p, temp_p = train_test_split(all_pairs, train_size=0.7, random_state=42)
val_p, test_p = train_test_split(temp_p, train_size=0.5, random_state=42)
print(f'Train: {len(train_p)} | Val: {len(val_p)} | Test: {len(test_p)}')


omnicrack30k found: 30017 pairs
masonry       : 240
crackforest   : 118
historical    : 0
omnicrack30k  : 3000
Total         : 3358
Train: 2350 | Val: 504 | Test: 504


In [7]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
_CLAHE = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))

def get_transforms(size=512):
    return A.Compose([A.Resize(size, size), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])

class SegDataset(Dataset):
    def __init__(self, pairs, size=512):
        self.pairs = pairs
        self.transform = get_transforms(size)
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        img_p, mask_p = self.pairs[idx]
        img = cv2.imread(str(img_p))
        img = np.zeros((512, 512, 3), dtype=np.uint8) if img is None else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        lab[..., 0] = _CLAHE.apply(lab[..., 0])
        img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        mask = cv2.imread(str(mask_p), cv2.IMREAD_GRAYSCALE)
        mask = np.zeros((512, 512), dtype=np.uint8) if mask is None else mask
        if mask.shape[:2] != img.shape[:2]:
            mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 127).astype(np.float32)
        aug = self.transform(image=img, mask=mask)
        return aug['image'], aug['mask'].unsqueeze(0)

test_loader = DataLoader(SegDataset(test_p, 512), batch_size=4, shuffle=False, num_workers=2)

In [9]:
import torch
import segmentation_models_pytorch as smp
from tqdm.notebook import tqdm
import numpy as np

DEVICE = torch.device('cuda')

ckpt = torch.load('/content/drive/MyDrive/HeritagePreservation/checkpoints/segmentor_v4/segmentor_best.pth', map_location=DEVICE, weights_only=False)
print(f'Loaded checkpoint: val_iou={ckpt["val_iou"]:.4f}, phase={ckpt["phase"]}, epoch={ckpt["epoch"]}')

model = smp.MAnet(encoder_name='mit_b2', encoder_weights=None, in_channels=3, classes=1, activation=None).to(DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()

print(f'Model: MAnet+mit_b2')
print(f'Val IoU (from ckpt): {ckpt["val_iou"]:.4f}')

def compute_metrics(pred_logits, target, threshold=0.5):
    pred_binary = (torch.sigmoid(pred_logits) > threshold).long()
    tp, fp, fn, tn = smp.metrics.get_stats(pred_binary, target.long(), mode='binary')
    return {
        'iou': float(smp.metrics.iou_score(tp, fp, fn, tn, reduction='micro')),
        'dice': float(smp.metrics.f1_score(tp, fp, fn, tn, reduction='micro')),
        'fp_rate': float(fp.sum().item() / (fp.sum().item() + tn.sum().item() + 1e-8)),
        'fn_rate': float(fn.sum().item() / (fn.sum().item() + tp.sum().item() + 1e-8)),
    }

Loaded checkpoint: val_iou=0.9762, phase=3, epoch=21
Model: MAnet+mit_b2
Val IoU (from ckpt): 0.9762


## Test: Threshold Tuning

In [10]:
print('\n=== Threshold Tuning (No Post-Processing) ===')
for threshold in [0.5, 0.55, 0.6, 0.65, 0.7]:
    test_iou, test_dice, test_fp, test_fn = [], [], [], []
    with torch.no_grad():
        for imgs, masks in tqdm(test_loader, desc=f'T={threshold}', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            pred = model(imgs)
            m = compute_metrics(pred, masks, threshold)
            test_iou.append(m['iou'])
            test_dice.append(m['dice'])
            test_fp.append(m['fp_rate'])
            test_fn.append(m['fn_rate'])
    print(f'  T={threshold:.2f}: IoU={np.mean(test_iou):.4f}  Dice={np.mean(test_dice):.4f}  FP={np.mean(test_fp):.4f}  FN={np.mean(test_fn):.4f}')


=== Threshold Tuning (No Post-Processing) ===


T=0.5:   0%|          | 0/126 [00:00<?, ?it/s]

  T=0.50: IoU=0.9742  Dice=0.9858  FP=0.3187  FN=0.0164


T=0.55:   0%|          | 0/126 [00:00<?, ?it/s]

  T=0.55: IoU=0.9747  Dice=0.9861  FP=0.3042  FN=0.0168


...

  T=0.60: IoU=0.9750  Dice=0.9863  FP=0.2897  FN=0.0172


T=0.65:   0%|          | 0/126 [00:00<?, ?it/s]

  T=0.65: IoU=0.9751  Dice=0.9864  FP=0.2741  FN=0.0178


T=0.7:   0%|          | 0/126 [00:00<?, ?it/s]

  T=0.70: IoU=0.9751  Dice=0.9864  FP=0.2527  FN=0.0187


## Post-Processing: Morphological Ops

In [12]:
def post_process(pred_logits, threshold=0.5, morph_kernel=5, min_size=50):
    pred = (torch.sigmoid(pred_logits) > threshold).cpu().numpy().astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morph_kernel, morph_kernel))
    result = []
    for m in pred:
        m = m.squeeze()
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, kernel)
        nl, labels, stats, _ = cv2.connectedComponentsWithStats(m)
        out = np.zeros_like(m)
        for i in range(1, nl):
            if stats[i, cv2.CC_STAT_AREA] >= min_size:
                out[labels == i] = 1
        result.append(torch.from_numpy(out[None]).float())
    return torch.cat(result, 0).unsqueeze(1)

print('Post-processing: morphological closing + min-size filter')
print('\n=== Threshold + Post-Processing ===')
for threshold in [0.5, 0.6, 0.7]:
    for min_sz in [30, 50]:
        test_iou, test_dice, test_fp, test_fn = [], [], [], []
        with torch.no_grad():
            for imgs, masks in tqdm(test_loader, desc=f'T={threshold} sz={min_sz}', leave=False):
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                pred = model(imgs)
                pred_pp = post_process(pred, threshold, min_size=min_sz).to(DEVICE)
                m = compute_metrics(pred_pp, masks, 0.5)
                test_iou.append(m['iou'])
                test_dice.append(m['dice'])
                test_fp.append(m['fp_rate'])
                test_fn.append(m['fn_rate'])
        print(f'  T={threshold:.1f} min_sz={min_sz:2d}: IoU={np.mean(test_iou):.4f}  FP={np.mean(test_fp):.4f}  FN={np.mean(test_fn):.4f}')


Post-processing: morphological closing + min-size filter

=== Threshold + Post-Processing ===


T=0.5 sz=30:   0%|          | 0/126 [00:00<?, ?it/s]

  T=0.5 min_sz=30: IoU=0.9741  FP=0.3268  FN=0.0163


T=0.5 sz=50:   0%|          | 0/126 [00:00<?, ?it/s]

  T=0.5 min_sz=50: IoU=0.9741  FP=0.3267  FN=0.0163


...

  T=0.6 min_sz=50: IoU=0.9749  FP=0.2946  FN=0.0171


T=0.7 sz=30:   0%|          | 0/126 [00:00<?, ?it/s]

  T=0.7 min_sz=30: IoU=0.9751  FP=0.2567  FN=0.0186


T=0.7 sz=50:   0%|          | 0/126 [00:00<?, ?it/s]

  T=0.7 min_sz=50: IoU=0.9751  FP=0.2566  FN=0.0186
